In [1]:
# rank_historians_openalex.py
#
# Install:
#   pip install requests pandas
#
# What it does:
#   1. Pulls topic-matching papers from OpenAlex
#   2. Keeps papers likely relevant to history / historiography
#   3. Builds paper-author edges
#   4. Scores and ranks authors as candidate "best historians"

import time
import math
import requests
import pandas as pd

# =========================================================
# CONFIG
# =========================================================

QUERY = '"historiography" OR "social history" OR "historical materialism" OR "feminist history" OR "history from below"'

MAX_PAPERS = 1000
BATCH_SIZE = 200
MIN_CITATION_COUNT = 5

HISTORY_KEYWORDS = [
    "history","historical","historiography","historian","historians",
    "archive","archival","memory","colonial","empire","imperial",
    "gender","feminist","labor","class","capitalism","revolution",
    "modernity","medieval","early modern","antiquity","nationalism",
    "subaltern","postcolonial","microhistory","annales","social history",
    "cultural history","intellectual history","political history",
    "economic history","oral history"
]

OUT_PAPERS = "topic_papers.csv"
OUT_EDGES = "paper_author_edges.csv"
OUT_RANKED = "ranked_historians.csv"

BASE_URL = "https://api.openalex.org/works"

# =========================================================
# HELPERS
# =========================================================

def log1p(x):
    try:
        return math.log1p(float(x or 0))
    except:
        return 0

def normalize_text(s):
    return (s or "").lower()

def paper_text_blob(p):

    parts=[
        p.get("title",""),
        p.get("abstract",""),
        " ".join([c.get("display_name","") for c in (p.get("concepts") or [])])
    ]

    return " ".join(parts).lower()

def history_relevance_score(p):

    blob=paper_text_blob(p)

    score=0

    for kw in HISTORY_KEYWORDS:
        if kw in blob:
            score+=1

    return score

# =========================================================
# 1) PULL PAPERS
# =========================================================

def pull_papers():

    papers=[]
    page=1

    while len(papers)<MAX_PAPERS:

        params={
            "search":QUERY,
            "per-page":BATCH_SIZE,
            "page":page
        }

        r=requests.get(BASE_URL,params=params)

        data=r.json()

        batch=data.get("results",[])

        if not batch:
            break

        papers.extend(batch)

        print("Fetched",len(batch),"papers; total =",len(papers))

        page+=1

        time.sleep(1)

    return papers[:MAX_PAPERS]

# =========================================================
# 2) FILTER HISTORY PAPERS
# =========================================================

def filter_history_papers(papers):

    kept=[]

    for p in papers:

        if p.get("cited_by_count",0) < MIN_CITATION_COUNT:
            continue

        score=history_relevance_score(p)

        if score>=2:
            kept.append(p)

    return kept

# =========================================================
# 3) BUILD EDGE TABLE
# =========================================================

def build_edges(filtered):

    paper_rows=[]
    edge_rows=[]

    for p in filtered:

        paper_rows.append({
            "paperId":p.get("id"),
            "title":p.get("title"),
            "year":p.get("publication_year"),
            "citationCount":p.get("cited_by_count")
        })

        authors=p.get("authorships") or []

        for idx,a in enumerate(authors):

            author=a.get("author",{})

            edge_rows.append({
                "paperId":p.get("id"),
                "title":p.get("title"),
                "year":p.get("publication_year"),
                "paperCitationCount":p.get("cited_by_count"),
                "history_relevance_score":history_relevance_score(p),
                "authorId":author.get("id"),
                "authorName":author.get("display_name"),
                "authorPosition":idx+1,
                "authorCount":len(authors),
                "isFirstAuthor":1 if idx==0 else 0,
                "isLastAuthor":1 if idx==(len(authors)-1) else 0
            })

    df_papers=pd.DataFrame(paper_rows).drop_duplicates()

    df_edges=pd.DataFrame(edge_rows)

    return df_papers,df_edges

# =========================================================
# 4) SCORE AUTHORS
# =========================================================

def score_authors(df_edges):

    grouped=(
        df_edges.groupby(["authorId","authorName"])
        .agg(
            matched_papers=("paperId","nunique"),
            total_topic_citations=("paperCitationCount","sum"),
            first_author_papers=("isFirstAuthor","sum"),
            last_author_papers=("isLastAuthor","sum"),
        )
        .reset_index()
    )

    grouped["score"]=(
        3*grouped["matched_papers"]
        +2*grouped["total_topic_citations"].map(log1p)
        +1*grouped["first_author_papers"].map(log1p)
        +1*grouped["last_author_papers"].map(log1p)
    )

    grouped=grouped.sort_values("score",ascending=False)

    grouped["rank"]=range(1,len(grouped)+1)

    return grouped

# =========================================================
# MAIN
# =========================================================

def main():

    print("1) Pulling papers...")
    papers=pull_papers()
    print("Retrieved",len(papers),"papers")

    print("2) Filtering history papers...")
    filtered=filter_history_papers(papers)
    print("Kept",len(filtered),"papers")

    print("3) Building edge tables...")
    df_papers,df_edges=build_edges(filtered)

    df_papers.to_csv(OUT_PAPERS,index=False)
    df_edges.to_csv(OUT_EDGES,index=False)

    print("4) Ranking authors...")
    ranked=score_authors(df_edges)

    ranked.to_csv(OUT_RANKED,index=False)

    print("\nTop historians:\n")
    print(ranked.head(25))

if __name__=="__main__":
    main()

1) Pulling papers...
Fetched 200 papers; total = 200
Fetched 200 papers; total = 400
Fetched 200 papers; total = 600
Fetched 200 papers; total = 800
Fetched 200 papers; total = 1000
Retrieved 1000 papers
2) Filtering history papers...
Kept 871 papers
3) Building edge tables...
4) Ranking authors...

Top historians:

                              authorId                 authorName  \
531   https://openalex.org/A5061949360         Arnaldo Momigliano   
242   https://openalex.org/A5025922453            Georg G. Iggers   
408   https://openalex.org/A5046659216             John Marincola   
751   https://openalex.org/A5086786755       Nancy C. M. Hartsock   
375   https://openalex.org/A5042827585         Joan Wallach Scott   
1014  https://openalex.org/A5113835045          F. M. L. Thompson   
768   https://openalex.org/A5088989714  George Macaulay Trevelyan   
942   https://openalex.org/A5111379049             E.F.K. Koerner   
892   https://openalex.org/A5109480611        Ellen Meiksins 

In [9]:
#getting sbtracts this time
# rank_historians_openalex_with_abstracts.py
#
# Install:
#   pip install requests pandas
#
# What it does:
#   1. Pulls topic-matching papers from OpenAlex
#   2. Keeps papers likely relevant to history / historiography
#   3. Builds paper-author edges
#   4. Scores and ranks authors as candidate "best historians"

import time
import math
import requests
import pandas as pd

# =========================================================
# CONFIG
# =========================================================

QUERY = '"historiography" OR "social history" OR "historical materialism" OR "feminist history" OR "history from below"'

MAX_PAPERS = 1000
BATCH_SIZE = 200
MIN_CITATION_COUNT = 5

HISTORY_KEYWORDS = [
    "history","historical","historiography","historian","historians",
    "archive","archival","memory","colonial","empire","imperial",
    "gender","feminist","labor","class","capitalism","revolution",
    "modernity","medieval","early modern","antiquity","nationalism",
    "subaltern","postcolonial","microhistory","annales","social history",
    "cultural history","intellectual history","political history",
    "economic history","oral history"
]

OUT_PAPERS = "topic_papers.csv"
OUT_EDGES = "paper_author_edges.csv"
OUT_RANKED = "ranked_historians.csv"

BASE_URL = "https://api.openalex.org/works"

# =========================================================
# HELPERS
# =========================================================

def log1p(x):
    try:
        return math.log1p(float(x or 0))
    except:
        return 0

def normalize_text(s):
    return (s or "").lower()

def reconstruct_abstract(inv_idx):
    """
    Reconstructs OpenAlex's abstract_inverted_index into plain text.
    """
    if not inv_idx:
        return ""
    # inv_idx: dict where keys are words, values are lists of positions
    positions = []
    for word, locs in inv_idx.items():
        for loc in locs:
            positions.append((loc, word))
    # sort by position
    positions.sort()
    words = [w for _, w in positions]
    return " ".join(words)

def paper_text_blob(p):
    parts=[
        p.get("title",""),
        reconstruct_abstract(p.get("abstract_inverted_index")),
        " ".join([c.get("display_name","") for c in (p.get("concepts") or [])])
    ]
    return " ".join(parts).lower()

def history_relevance_score(p):
    blob=paper_text_blob(p)
    score=0
    for kw in HISTORY_KEYWORDS:
        if kw in blob:
            score+=1
    return score

# =========================================================
# 1) PULL PAPERS
# =========================================================

def pull_papers():
    papers=[]
    page=1
    while len(papers)<MAX_PAPERS:
        params={
            "search":QUERY,
            "per-page":BATCH_SIZE,
            "page":page
        }
        r=requests.get(BASE_URL,params=params)
        data=r.json()
        batch=data.get("results",[])
        if not batch:
            break
        papers.extend(batch)
        print("Fetched",len(batch),"papers; total =",len(papers))
        page+=1
        time.sleep(1)
    return papers[:MAX_PAPERS]

# =========================================================
# 2) FILTER HISTORY PAPERS
# =========================================================

def filter_history_papers(papers):
    kept=[]
    for p in papers:
        if p.get("cited_by_count",0) < MIN_CITATION_COUNT:
            continue
        score=history_relevance_score(p)
        if score>=2:
            kept.append(p)
    return kept

# =========================================================
# 3) BUILD EDGE TABLE
# =========================================================

def build_edges(filtered):
    paper_rows=[]
    edge_rows=[]
    for p in filtered:
        paper_rows.append({
            "paperId":p.get("id"),
            "title":p.get("title"),
            "abstract":reconstruct_abstract(p.get("abstract_inverted_index")),
            "year":p.get("publication_year"),
            "citationCount":p.get("cited_by_count")
        })
        authors=p.get("authorships") or []
        for idx,a in enumerate(authors):
            author=a.get("author",{})
            edge_rows.append({
                "paperId":p.get("id"),
                "title":p.get("title"),
                "year":p.get("publication_year"),
                "paperCitationCount":p.get("cited_by_count"),
                "history_relevance_score":history_relevance_score(p),
                "authorId":author.get("id"),
                "authorName":author.get("display_name"),
                "authorPosition":idx+1,
                "authorCount":len(authors),
                "isFirstAuthor":1 if idx==0 else 0,
                "isLastAuthor":1 if idx==(len(authors)-1) else 0
            })
    df_papers=pd.DataFrame(paper_rows).drop_duplicates()
    df_edges=pd.DataFrame(edge_rows)
    return df_papers,df_edges

# =========================================================
# 4) SCORE AUTHORS
# =========================================================

def score_authors(df_edges):
    grouped=(
        df_edges.groupby(["authorId","authorName"])
        .agg(
            matched_papers=("paperId","nunique"),
            total_topic_citations=("paperCitationCount","sum"),
            first_author_papers=("isFirstAuthor","sum"),
            last_author_papers=("isLastAuthor","sum"),
        )
        .reset_index()
    )
    grouped["score"]=(
        3*grouped["matched_papers"]
        +2*grouped["total_topic_citations"].map(log1p)
        +1*grouped["first_author_papers"].map(log1p)
        +1*grouped["last_author_papers"].map(log1p)
    )
    grouped=grouped.sort_values("score",ascending=False)
    grouped["rank"]=range(1,len(grouped)+1)
    return grouped

# =========================================================
# MAIN
# =========================================================

def main():
    print("1) Pulling papers...")
    papers=pull_papers()
    print("Retrieved",len(papers),"papers")

    print("2) Filtering history papers...")
    filtered=filter_history_papers(papers)
    print("Kept",len(filtered),"papers")

    print("3) Building edge tables...")
    df_papers,df_edges=build_edges(filtered)
    df_papers.to_csv(OUT_PAPERS,index=False, encoding="utf-8-sig")
    df_edges.to_csv(OUT_EDGES,index=False, encoding="utf-8-sig")

    print("4) Ranking authors...")
    ranked=score_authors(df_edges)
    ranked.to_csv(OUT_RANKED,index=False, encoding="utf-8-sig")

    print("\nTop historians:\n")
    print(ranked.head(25))

if __name__=="__main__":
    main()

1) Pulling papers...
Fetched 200 papers; total = 200
Fetched 200 papers; total = 400
Fetched 200 papers; total = 600
Fetched 200 papers; total = 800
Fetched 200 papers; total = 1000
Retrieved 1000 papers
2) Filtering history papers...
Kept 950 papers
3) Building edge tables...
4) Ranking authors...

Top historians:

                              authorId                 authorName  \
580   https://openalex.org/A5061949360         Arnaldo Momigliano   
265   https://openalex.org/A5025922453            Georg G. Iggers   
444   https://openalex.org/A5046659216             John Marincola   
815   https://openalex.org/A5086786755       Nancy C. M. Hartsock   
996   https://openalex.org/A5110330477            Clifford Geertz   
410   https://openalex.org/A5042827585         Joan Wallach Scott   
1097  https://openalex.org/A5113835045          F. M. L. Thompson   
555   https://openalex.org/A5059088641         Reinhart Koselleck   
832   https://openalex.org/A5088989714  George Macaulay Treve

In [11]:
import pandas as pd

# =========================================================
# Load CSV outputs
# =========================================================
papers_df = pd.read_csv("topic_papers.csv")
edges_df = pd.read_csv("paper_author_edges.csv")
ranked_df = pd.read_csv("ranked_historians.csv")

# Pick top 25 authors
top_authors = ranked_df.head(25)

# =========================================================
# Helper: build persona prompt from papers
# =========================================================
def build_persona_prompt(historian_name, papers):
    """
    Build a persona from a historian's actual published work.
    """
    # Concatenate all abstracts
    full_corpus = "\n\n".join([
        f"Title: {p['title']} ({p['year']})\nAbstract: {p['abstract']}"
        for p in papers
        if p.get("abstract")
    ])

    prompt = f"""You are a historian. Your scholarly perspective is defined
entirely by the following body of prior work — these are real publications
that characterize your intellectual position, methodological commitments,
and interpretive frameworks:

{full_corpus}

When engaging in historical research and debate:
- Reason from the methodological approach evident in this prior work
- Prioritize the kinds of sources and evidence this work relies on
- Make arguments in the style and register of this scholarship
- Do not announce or describe your perspective explicitly — embody it
- Your name in this conversation is {historian_name}
"""
    return prompt

# =========================================================
# Build persona prompts for top 25 historians
# =========================================================
historian_prompts = {}

for _, row in top_authors.iterrows():
    author_id = row["authorId"]
    author_name = row["authorName"]

    # Get all paperIds for this historian
    paper_ids = edges_df.loc[edges_df["authorId"] == author_id, "paperId"].unique()

    # Get the actual papers
    papers = papers_df.loc[papers_df["paperId"].isin(paper_ids)]

    # Build the persona prompt
    historian_prompts[author_name] = build_persona_prompt(author_name, papers.to_dict("records"))

# =========================================================
# Optional: preview first 500 chars of each persona
# =========================================================
for name, prompt in historian_prompts.items():
    print(f"=== Persona: {name} ===")
    print(prompt[:500], "...\n")

=== Persona: Arnaldo Momigliano ===
You are a historian. Your scholarly perspective is defined 
entirely by the following body of prior work — these are real publications 
that characterize your intellectual position, methodological commitments, 
and interpretive frameworks:

Title: Time in Ancient Historiography (1966)
Abstract: Before I get down to those problems about time that seem to me relevant to ancient historiography, I have to eliminate a series of faulty questions, part of which at least derive from an arbitrary introdu ...

=== Persona: Georg G. Iggers ===
You are a historian. Your scholarly perspective is defined 
entirely by the following body of prior work — these are real publications 
that characterize your intellectual position, methodological commitments, 
and interpretive frameworks:

Title: Historiography in the Twentieth Century: From Scientific Objectivity to the Postmodern Challenge (1998)
Abstract: A preeminent intellectual historian here examines the profound 

In [26]:
print(papers_df.columns.tolist())
print(papers_df.head(2))

['paperId', 'title', 'abstract', 'year', 'citationCount']
                            paperId  \
1  https://openalex.org/W2162874368   
3  https://openalex.org/W1572168882   

                                               title  \
1  The Feminist Standpoint: Developing the Ground...   
3  Centuries of Childhood. A Social History of Fa...   

                                            abstract  year  citationCount  
1  The power of the Marxian critique of class dom...  2005           1446  
3  What was the role of the Catholic colleges in ...  1963           2498  


In [27]:
for name, prompt in historian_prompts.items():
    print(f"=== {name} ===")
    print(f"Prompt length: {len(prompt)} characters")
    print(prompt[:300])
    print()

=== Arnaldo Momigliano ===
Prompt length: 5869 characters
You are a historian. Your scholarly perspective is defined 
entirely by the following body of prior work — these are real publications 
that characterize your intellectual position, methodological commitments, 
and interpretive frameworks:

Title: Time in Ancient Historiography (1966)
Abstract: Befo

=== Georg G. Iggers ===
Prompt length: 6166 characters
You are a historian. Your scholarly perspective is defined 
entirely by the following body of prior work — these are real publications 
that characterize your intellectual position, methodological commitments, 
and interpretive frameworks:

Title: Historiography in the Twentieth Century: From Scient

=== John Marincola ===
Prompt length: 3740 characters
You are a historian. Your scholarly perspective is defined 
entirely by the following body of prior work — these are real publications 
that characterize your intellectual position, methodological commitments, 
and interpretive f

In [17]:
# How many rows and columns
print(papers_df.shape)  # (rows, columns)

# Quick info about columns, types, and non-null counts
print(papers_df.info())

# Number of missing values per column
print(papers_df["abstract"].isna().sum())

# Percentage of missing values per column
print(papers_df['abstract'].isna().mean() * 100)


(950, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   paperId        950 non-null    object
 1   title          950 non-null    object
 2   abstract       720 non-null    object
 3   year           950 non-null    int64 
 4   citationCount  950 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 37.2+ KB
None
230
24.210526315789473


In [22]:
# Drop rows where 'abstract' is NaN
papers_df = papers_df.dropna(subset=['abstract'])

# Check the result
print(papers_df.shape)
print(papers_df['abstract'].isna().sum())  # should be 0

(720, 5)
0


In [24]:
papers_df.iloc[0]['abstract']

'The power of the Marxian critique of class domination stands as an implicit suggestion that feminists should consider the advantages of adopting a historical materialist approach to understanding phallocratic domination. A specifically feminist historical materialism might enable us to lay bare the laws of tendency which constitute the structure of patriarchy over time and to follow its development in and through the Western class societies on which Marx’s interest centered. A feminist materialism might in addition enable us to expand the Marxian account to include all human activity rather than focussing on activity more characteristic of males in capitalism. The development of such a historical and materialist account is a very large task, one which requires the political and theoretical contributions of many feminists. Here I will address only the question of the epistemological underpinnings such a materialism would require. Most specifically, I will attempt to develop, on the met

In [25]:
papers_df.head()

,paperId,title,abstract,year,citationCount
1,https://openalex.org/W2162874368,The Feminist Standpoint: Developing the Ground...,The power of the Marxian critique of class dom...,2005,1446
3,https://openalex.org/W1572168882,Centuries of Childhood. A Social History of Fa...,What was the role of the Catholic colleges in ...,1963,2498
4,https://openalex.org/W1527454661,A Contemporary Critique of Historical Materialism,Preface to the Second Edition - Introduction -...,1995,1203
5,https://openalex.org/W2170667996,Discovering the News: A Social History of Amer...,Journal Article Discovering the News: A Social...,1979,1467
7,https://openalex.org/W619048631,Habermas on historical materialism,Introduction I. Theory Reconstruction and Theo...,1989,235


In [4]:
edges_df.head()

,paperId,title,year,paperCitationCount,history_relevance_score,authorId,authorName,authorPosition,authorCount,isFirstAuthor,isLastAuthor
0,https://openalex.org/W2045633722,"Trust, Reciprocity, and Social History",1995,5560,2,https://openalex.org/A5021459093,Joyce E. Berg,1,3,1,0
1,https://openalex.org/W2045633722,"Trust, Reciprocity, and Social History",1995,5560,2,https://openalex.org/A5062148925,John Dickhaut,2,3,0,0
2,https://openalex.org/W2045633722,"Trust, Reciprocity, and Social History",1995,5560,2,https://openalex.org/A5076630423,Kevin McCabe,3,3,0,1
3,https://openalex.org/W2162874368,The Feminist Standpoint: Developing the Ground...,2005,1446,5,https://openalex.org/A5086786755,Nancy C. M. Hartsock,1,1,1,1
4,https://openalex.org/W4298054234,A Contemporary Critique of Historical Materialism,1981,1522,2,https://openalex.org/A5034980949,Anthony Giddens,1,1,1,1
